# Small VLMs & Multi-Image Workflows — Practical Vision at Scale

> **Orbit 5 (Multimodal)** · Notebook 8 of 22

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/08_small_vlms_and_multi_image_workflows.ipynb)

## Learning Objectives

By the end of this notebook you will be able to:

- **Understand the model-size trade-off** — 7B vs 3B vs 2B VLMs and when smaller is better
- **Implement multi-image inputs** for comparing, sequencing, and batch-processing images
- **Build a practical multi-page document workflow** that treats PDF pages as images
- **Measure and compare** VRAM usage, latency, and quality across model sizes
- **Design routing strategies** that pick the right-sized model for each task

## 1 · The Model Size Trade-Off

Bigger is not always better — especially when you are paying per token, per GPU-second,
per watt. The Qwen2.5-VL family ships in three sizes: **2B, 7B, and 72B**. Each step up
roughly quadruples the compute required, so the jump from 2B to 7B is not merely
"a little more expensive"; it is a fundamentally different cost tier.

What do you **lose** with smaller models? Complex multi-step reasoning, rare-knowledge
recall, and nuanced chain-of-thought explanations all degrade. What do you **keep**?
Basic object recognition, OCR, simple descriptions, and classification hold up
remarkably well even at 2–3 B parameters.

The **80/20 rule** applies heavily here. For many production tasks — receipt OCR,
product-photo classification, simple visual Q&A — a 2B model achieves 80–90 % of the
quality of a 7B model at roughly 3× the throughput. On a T4 GPU the 7B model quantised
to 4-bit fits in about 5 GB of VRAM; the 2B-4bit variant needs only ≈ 2 GB, leaving
headroom for other services on the same card.

Before reaching for the biggest model, ask four questions: **(1)** What accuracy do I
need? **(2)** What is my latency budget? **(3)** What is my VRAM budget? **(4)** How many
requests per second must I sustain? The answers almost always point to something
smaller than the flagship.

In this notebook we will load both the 7B and 3B Qwen2.5-VL variants, run identical
tasks on each, and measure the differences in quality, latency, and memory.

In [ ]:
!pip install -q "transformers>=4.57.0" accelerate bitsandbytes torch pillow qwen-vl-utils requests pandas matplotlib

In [ ]:
import torch, json, time, gc
from PIL import Image, ImageDraw, ImageFont
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import pandas as pd
import numpy as np

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

# Load the 7B model
MODEL_7B = "Qwen/Qwen2.5-VL-7B-Instruct"
model_7b = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_7B, quantization_config=bnb, device_map="auto"
)
processor_7b = AutoProcessor.from_pretrained(MODEL_7B)
vram_7b = torch.cuda.memory_allocated() / 1024**3
print(f"✓ 7B loaded | VRAM: {vram_7b:.1f} GB")

### Helper — single-image inference

A thin wrapper that builds the chat template, tokenises, generates, and decodes.

In [ ]:
def ask_vlm(image, prompt, model, processor, max_tokens=512):
    """Send a single image + prompt to a Qwen2.5-VL model and return the text response."""
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(
        text=[text], images=[image], return_tensors="pt", padding=True
    ).to(model.device)
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=max_tokens)
    return processor.batch_decode(
        ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
    )[0]

### Synthetic test images

We create simple product cards so the notebook is self-contained and runs without
downloading external data.

In [ ]:
def make_product_image(name, price, color):
    img = Image.new("RGB", (300, 300), "white")
    draw = ImageDraw.Draw(img)
    draw.rectangle([50, 30, 250, 200], fill=color, outline="black", width=2)
    draw.text((60, 210), name, fill="black")
    draw.text((60, 235), f"${price:.2f}", fill="darkgreen")
    draw.text((60, 260), "In Stock", fill="gray")
    return img

products = [
    make_product_image("Laptop Pro X1", 1299.99, "silver"),
    make_product_image("Wireless Mouse", 29.99, "darkblue"),
    make_product_image("USB-C Hub", 49.99, "gray"),
]

# Display grid
grid = Image.new("RGB", (920, 300), "white")
for i, p in enumerate(products):
    grid.paste(p, (i * 310, 0))
grid

## 2 · Benchmarking Model Sizes

To make informed decisions about which model to deploy, we need **measurements**, not
intuition. We evaluate along three dimensions:

| Dimension | What it measures | Why it matters |
|-----------|-----------------|----------------|
| **Quality** | How good are the answers? | Directly affects user experience |
| **Latency** | Seconds per inference | Determines UX responsiveness |
| **VRAM** | GPU memory consumed | Limits co-tenancy and batch size |

Quality differences are **task-dependent**. For straightforward OCR the gap between 3B
and 7B is often negligible. For multi-step reasoning or counting objects the larger
model pulls ahead significantly. The only reliable way to know is to benchmark on
*your* workload.

In [ ]:
def benchmark_model(model, processor, images, prompts, task_names, label):
    results = []
    for img, prompt, task_name in zip(images, prompts, task_names):
        torch.cuda.synchronize()
        t0 = time.time()
        response = ask_vlm(img, prompt, model, processor, max_tokens=200)
        torch.cuda.synchronize()
        elapsed = time.time() - t0
        results.append({
            "model": label,
            "task": task_name,
            "response": response.strip()[:100],
            "latency_s": round(elapsed, 2),
            "vram_gb": round(torch.cuda.memory_allocated() / 1024**3, 1),
        })
    return results

test_scene = make_product_image("Test Widget", 19.99, "coral")
benchmark_images = [test_scene, test_scene, test_scene]
benchmark_prompts = [
    "What product is shown? Reply in one line.",
    "What is the price of this product? Reply with just the price.",
    "Describe this image in detail.",
]
task_names = ["recognition", "extraction", "description"]

results_7b = benchmark_model(
    model_7b, processor_7b, benchmark_images, benchmark_prompts, task_names, "7B"
)
df_7b = pd.DataFrame(results_7b)
print(df_7b.to_string(index=False))

### Swap models — unload 7B, load 3B

We explicitly delete the model and flush the CUDA cache so the 3B model starts from
a clean memory state.

In [ ]:
del model_7b
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after unload: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

MODEL_3B = "Qwen/Qwen2.5-VL-3B-Instruct"
model_3b = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_3B, quantization_config=bnb, device_map="auto"
)
processor_3b = AutoProcessor.from_pretrained(MODEL_3B)
vram_3b = torch.cuda.memory_allocated() / 1024**3
print(f"✓ 3B loaded | VRAM: {vram_3b:.1f} GB")
print(f"VRAM savings: {vram_7b - vram_3b:.1f} GB ({(1 - vram_3b / vram_7b) * 100:.0f}% less)")

In [ ]:
results_3b = benchmark_model(
    model_3b, processor_3b, benchmark_images, benchmark_prompts, task_names, "3B"
)
df_3b = pd.DataFrame(results_3b)
print(df_3b.to_string(index=False))

In [ ]:
df_all = pd.concat([df_7b, df_3b])
comparison = df_all.pivot_table(
    index="task", columns="model",
    values=["latency_s", "vram_gb"], aggfunc="first"
)
print("═══ Model Comparison ═══")
print(comparison.to_string())
print(f"\nAvg latency — 7B: {df_7b['latency_s'].mean():.2f}s, "
      f"3B: {df_3b['latency_s'].mean():.2f}s")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Latency comparison
tasks = df_7b["task"].tolist()
lat_7b = df_7b["latency_s"].tolist()
lat_3b_vals = df_3b["latency_s"].tolist()
x = np.arange(len(tasks))
ax1.bar(x - 0.2, lat_7b, 0.35, label="7B", color="steelblue")
ax1.bar(x + 0.2, lat_3b_vals, 0.35, label="3B", color="coral")
ax1.set_xticks(x)
ax1.set_xticklabels(tasks)
ax1.set_ylabel("Latency (s)")
ax1.set_title("Latency by Model Size")
ax1.legend()

# VRAM comparison
ax2.bar(["7B", "3B"], [vram_7b, vram_3b], color=["steelblue", "coral"])
ax2.set_ylabel("VRAM (GB)")
ax2.set_title("VRAM Usage")

plt.tight_layout()
plt.show()

## 3 · Multi-Image Inputs

Many real-world tasks involve **more than one image**: comparing two product photos,
sequencing frames from a video, or processing multiple pages of a document in a single
call. Qwen2.5-VL natively supports multi-image inputs — you simply list several image
entries in the user message.

Each image adds **visual tokens** to the context window. Depending on resolution and
the model's internal tiling strategy, one image may contribute 256–1 280 tokens. Five
images could therefore cost 1 000–6 000 tokens before you even write the prompt,
so **budget carefully**.

Common multi-image prompting patterns:

| Pattern | Example prompt |
|---------|---------------|
| **Sequential description** | "Describe each image in order." |
| **Comparison** | "What's different between image 1 and image 2?" |
| **Synthesis** | "Summarise all the documents shown." |

Image **order** in the prompt matters: the model processes them left-to-right, and
early images have slightly more influence on the generated text.

In [ ]:
def ask_vlm_multi(images, prompt, model, processor, max_tokens=512):
    """Send multiple images + a single prompt to the VLM."""
    content = [{"type": "image", "image": img} for img in images]
    content.append({"type": "text", "text": prompt})
    messages = [{"role": "user", "content": content}]
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[text], images=list(images), return_tensors="pt", padding=True
    ).to(model.device)
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=max_tokens)
    return processor.batch_decode(
        ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
    )[0]

In [ ]:
print("═══ Multi-Image: Product Comparison ═══\n")
response = ask_vlm_multi(
    products[:2],
    "Compare these two products. Which is more expensive? "
    "What are the key differences?",
    model_3b, processor_3b
)
print(response)

In [ ]:
print("═══ Multi-Image: All Three Products ═══\n")
response = ask_vlm_multi(
    products,
    "These are three product images. List each product with its name and price, "
    "then rank them from most to least expensive.",
    model_3b, processor_3b
)
print(response)

## 4 · Multi-Page Document Workflow

Many business documents span multiple pages: contracts, quarterly reports, research
papers. The standard strategy is to **render each page as an image** and feed them to
the VLM.

Two processing approaches:

1. **Page-by-page** — process each page independently, then aggregate the summaries.
   More reliable because each call has a small, focused context.
2. **Multi-page-in-one-call** — pass all page images in a single prompt and ask the
   model to synthesise. Richer cross-page reasoning, but riskier because the long
   context may cause the model to hallucinate or drop details.

In practice, a **hybrid** works best: summarise each page individually, then pass the
text summaries (not the images) to a second LLM call for final synthesis.

In [ ]:
def make_document_pages():
    pages = []
    # Page 1: Title page
    p1 = Image.new("RGB", (600, 800), "white")
    d1 = ImageDraw.Draw(p1)
    d1.text((150, 200), "QUARTERLY REPORT", fill="black")
    d1.text((200, 250), "Q4 2024", fill="gray")
    d1.text((150, 350), "Acme Corporation", fill="black")
    d1.text((150, 380), "Prepared: January 2025", fill="gray")
    d1.text((550, 760), "Page 1", fill="gray")
    pages.append(p1)

    # Page 2: Financial summary
    p2 = Image.new("RGB", (600, 800), "white")
    d2 = ImageDraw.Draw(p2)
    d2.text((30, 20), "Financial Summary", fill="black")
    d2.line([(30, 45), (570, 45)], fill="black")
    lines = [
        "Revenue: $4.2M (+12% YoY)",
        "Expenses: $3.1M (+5% YoY)",
        "Net Income: $1.1M (+32% YoY)",
        "Cash: $2.8M",
        "Headcount: 47 (+8)",
    ]
    y = 65
    for line in lines:
        d2.text((50, y), line, fill="black")
        y += 35
    d2.text((550, 760), "Page 2", fill="gray")
    pages.append(p2)

    # Page 3: Outlook
    p3 = Image.new("RGB", (600, 800), "white")
    d3 = ImageDraw.Draw(p3)
    d3.text((30, 20), "Q1 2025 Outlook", fill="black")
    d3.line([(30, 45), (570, 45)], fill="black")
    outlook = [
        "- Launch Product v2.0 in February",
        "- Expand to European market",
        "- Target revenue: $4.8M",
        "- Hiring plan: 5 engineers, 2 sales",
    ]
    y = 65
    for line in outlook:
        d3.text((50, y), line, fill="black")
        y += 35
    d3.text((550, 760), "Page 3", fill="gray")
    pages.append(p3)
    return pages

doc_pages = make_document_pages()

# Show all pages side by side
combined = Image.new("RGB", (1830, 800), "lightgray")
for i, page in enumerate(doc_pages):
    combined.paste(page, (i * 610, 0))
combined

In [ ]:
print("═══ Page-by-Page Processing ═══\n")
page_summaries = []
for i, page in enumerate(doc_pages):
    t0 = time.time()
    summary = ask_vlm(
        page,
        "Summarize this page in 2-3 sentences. What are the key facts?",
        model_3b, processor_3b, max_tokens=150,
    )
    elapsed = time.time() - t0
    page_summaries.append({
        "page": i + 1,
        "summary": summary.strip(),
        "time_s": round(elapsed, 2),
    })
    print(f"Page {i + 1} ({elapsed:.1f}s): {summary.strip()}\n")

df_pages = pd.DataFrame(page_summaries)
print(f"Total processing time: {df_pages['time_s'].sum():.1f}s")

In [ ]:
print("═══ Multi-Page Synthesis ═══\n")
response = ask_vlm_multi(
    doc_pages,
    "These are 3 pages of a quarterly report. Provide a complete executive "
    "summary covering: company name, quarter, key financial metrics, and "
    "outlook. Be specific with numbers.",
    model_3b, processor_3b, max_tokens=400,
)
print(response)

## 5 · Cost Analysis

Every image adds tokens, and tokens cost money (or GPU-seconds on your own hardware).
Understanding the token budget is critical for cost control.

**Rule of thumb:** each image contributes roughly 256–1 280 visual tokens, depending
on resolution and the model's internal tiling. For a batch of *N* images the total
token count is approximately:

```
total ≈ N × image_tokens + prompt_tokens + output_tokens
```

Latency scales roughly linearly with context length, while VRAM is **fixed** by the
model weights (assuming sequential processing, not dynamic batching). When processing
large batches, prefer **sequential single-image calls** for predictable latency over
cramming many images into one context window.

In [ ]:
def estimate_cost(n_images, tokens_per_image=500, prompt_tokens=50,
                  output_tokens=200, cost_per_1k_tokens=0.003):
    total_input = n_images * tokens_per_image + prompt_tokens
    total_output = output_tokens
    cost = (total_input + total_output) * cost_per_1k_tokens / 1000
    return {
        "n_images": n_images,
        "input_tokens": total_input,
        "output_tokens": total_output,
        "total_tokens": total_input + total_output,
        "est_cost_usd": round(cost, 4),
    }

scenarios = [estimate_cost(1), estimate_cost(5),
             estimate_cost(20), estimate_cost(100)]
df_cost = pd.DataFrame(scenarios)
print("═══ Cost Estimation ═══")
print(df_cost.to_string(index=False))
print(f"\n100 images at API pricing ≈ ${scenarios[-1]['est_cost_usd']:.2f}")

In [ ]:
ns = [1, 2, 5, 10, 20, 50, 100]
costs = [estimate_cost(n)["est_cost_usd"] for n in ns]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ns, costs, "o-", color="steelblue", linewidth=2)
ax.set_xlabel("Number of images")
ax.set_ylabel("Estimated cost (USD)")
ax.set_title("Cost Scaling with Image Count")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6 · When to Use VLMs vs Dedicated CV Models

VLMs are **general-purpose**: they can handle almost any visual question, but they
rarely beat a specialist. Dedicated models — YOLO for detection, ResNet for
classification, EasyOCR for text extraction — are faster, cheaper, and often more
accurate on their specific task.

| Scenario | Best tool | Why |
|----------|-----------|-----|
| Classify product photos | Dedicated CNN | Fast, consistent, easy to fine-tune |
| Detect all faces in a crowd | YOLO / RetinaFace | Real-time, high recall |
| Read text from a form | EasyOCR / Tesseract | Optimised for text, no hallucination |
| Explain a medical image | VLM (7B+) | Needs reasoning and domain knowledge |
| Compare two product photos | VLM (3B) | Flexible, handles open-ended queries |

The **hybrid approach** is often the winner: use a cheap, fast classifier to triage
incoming images, then route only the complex cases to the VLM. This combines the
throughput of dedicated models with the flexibility of VLMs.

In [ ]:
def route_task(task_description):
    """Simple routing logic: pick the right tool for the job."""
    task = task_description.lower()
    if any(k in task for k in ["classify", "is this a", "what type"]):
        return "dedicated_classifier", "Fast CNN classifier"
    elif any(k in task for k in ["detect", "find all", "count", "locate"]):
        return "dedicated_detector", "YOLO or similar"
    elif any(k in task for k in ["ocr", "read text", "extract text"]):
        return "dedicated_ocr", "EasyOCR / Tesseract"
    elif any(k in task for k in ["describe", "explain", "compare", "reason", "why"]):
        return "vlm_large", "7B+ VLM needed"
    else:
        return "vlm_small", "3B VLM sufficient"

test_tasks = [
    "Classify this image as food or not-food",
    "Find all cars in this parking lot image",
    "Read the text on this business card",
    "Explain what is happening in this medical image",
    "What color is the main object?",
]

print("═══ Task Routing Demo ═══\n")
for task in test_tasks:
    route, tool = route_task(task)
    print(f"  {task:50s} → {route:22s} ({tool})")

### Batch processing with routing

In production you would wire the router to actual model endpoints. Here we simulate
the dispatching logic and show how routing reduces overall compute.

In [ ]:
def simulate_batch_processing(tasks, images):
    """Simulate routed processing with timing estimates."""
    log = []
    for task, img in zip(tasks, images):
        route, tool = route_task(task)
        if route.startswith("dedicated"):
            est_time = 0.05
        elif route == "vlm_small":
            est_time = 1.5
        else:
            est_time = 4.0
        log.append({"task": task[:40], "route": route, "est_time_s": est_time})
    return pd.DataFrame(log)

batch_tasks = [
    "Classify this product photo",
    "Read the text from this receipt",
    "Describe what is unusual in this image",
    "What type of food is shown?",
    "Count the number of items on the shelf",
]
batch_images = [products[0]] * len(batch_tasks)

df_batch = simulate_batch_processing(batch_tasks, batch_images)
print("═══ Batch Routing Results ═══")
print(df_batch.to_string(index=False))
routed_total = df_batch["est_time_s"].sum()
naive_total = len(batch_tasks) * 4.0
print(f"\nRouted total: {routed_total:.1f}s vs Naive (all 7B): {naive_total:.1f}s")
print(f"Speedup: {naive_total / routed_total:.1f}×")

## 7 · Exercises

**Exercise 1 — Quality benchmark.** Load both the 7B and 3B models and run a
10-question benchmark mixing easy tasks (OCR, simple description) with hard tasks
(reasoning, counting). Score each answer manually (0/1) and plot accuracy vs model
size. Does the gap widen on hard questions?

**Exercise 2 — Structured document extraction.** Create a 5-page synthetic document
(invoice, contract, or report). For each page, extract the key information into a
structured JSON object (e.g., `{"page": 1, "title": "...", "key_facts": [...]}`).
Combine the per-page JSONs into a single document summary.

**Exercise 3 — Smart routing pipeline.** Implement the routing strategy end-to-end:
classify each incoming image-task pair into easy / medium / hard, route easy tasks to
the 3B model and hard tasks to the 7B, and measure wall-clock throughput vs sending
everything to the 7B.

In [ ]:
# ── Exercise 1 starter ──

exercise_questions = [
    # (image, prompt, task_difficulty)
    (make_product_image("Widget A", 9.99, "red"), "What is the price?", "easy"),
    (make_product_image("Widget B", 24.50, "blue"), "What product is shown?", "easy"),
    (make_product_image("Widget C", 5.00, "green"), "Is this product expensive or cheap?", "hard"),
    # Add 7 more questions to reach 10 ...
]

# Skeleton: iterate, score, plot
for img, prompt, difficulty in exercise_questions:
    response = ask_vlm(img, prompt, model_3b, processor_3b, max_tokens=100)
    print(f"[{difficulty}] {prompt} → {response.strip()[:60]}")

## Key Takeaways

1. **Smaller VLMs (2–3 B) achieve 80–90 % of large-model quality** on simple tasks
   (OCR, classification, basic description) at 2–3× the speed and half the VRAM.
2. **Multi-image workflows multiply token costs** — budget carefully using
   tokens-per-image estimates (256–1 280 tokens each).
3. **Page-by-page processing is more reliable** than multi-page-in-one-call for long
   documents; use text-level aggregation for cross-page synthesis.
4. **Smart routing** (small model for easy tasks, large model for hard) gives the best
   cost-quality trade-off in production.
5. **Always benchmark YOUR tasks on YOUR data** — public leaderboards do not predict
   your specific accuracy or latency.

## References

- Qwen2.5-VL Technical Report (2024) — Alibaba DAMO Academy
- "Scaling Vision-Language Models" — survey of efficiency techniques (2023-2024)
- "MobileVLM: A Fast, Strong and Open Vision Language Assistant" — Chu et al. (2024)
- Castalia notebooks `multimodal/05`–`07` for single-image VLM foundations

---

[← 07 Structured Outputs, Grounding & Localization](07_structured_outputs_grounding_and_localization.ipynb) | [09 Captioning as a Text Bridge →](09_captioning_as_a_text_bridge.ipynb)